# ATOM Wind — Grand Tunis ET Tunisie entière en ligne (OS5)

Un seul service sert **deux domaines** : le modèle Grand Tunis livré (inchangé) et le **modèle national C' s44** (six DSO conformes sur le test 2022, w servi β = 0,75), avec la **couche 10 m corrigée par les aéroports** (`regle_produit_vent10m_v4`). Le domaine est choisi par la position (Tunis → Grand Tunis ; Sfax, Gafsa, Tozeur, Gabès → national). Le carnet mesure le **DSO4 sur le GPU du service** puis fait la **démonstration** sur 5 villes.

**Avant** : secret Colab `TUNNEL_CRED_CONTENTS` (remis par Rayen), « Accès au notebook » activé. **GPU** : L4 (inférence, ~2 unités/h). Arrêt : `ARRETER = True` dans la dernière cellule.


## 1. Drive, code et données nationales

In [ ]:
from google.colab import drive
import os as _os
if not _os.path.isdir("/content/drive/MyDrive"):   # deja monte (panneau Fichiers) : ne pas redemander
    drive.mount("/content/drive")

import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

GT = Path("/content/drive/MyDrive/pcfno3d_grand_tunis")   # code et poids du Grand Tunis
NAT = Path("/content/drive/MyDrive/pcfno3d_national")
NAT.mkdir(parents=True, exist_ok=True)
JOURNAUX = NAT / "journaux"
JOURNAUX.mkdir(exist_ok=True)
libre = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
print(f"place libre sur le Drive : {libre:.0f} Go (le jeu national en demande ~50)")

CODE = Path("/content/PFE")
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
with zipfile.ZipFile(GT / "pcfno3d_code.zip") as zf:
    zf.extractall(CODE)
tampon = CODE / "PCFNO3D_VERSION.txt"
print(tampon.read_text().strip() if tampon.exists() else "tampon : absent")
%cd /content/PFE
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q pandera xarray netCDF4 h5netcdf 2>&1 | tail -2
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]

def lancer(nom: str, commande: str) -> None:
    """Commande obligatoire ; sortie affichee ET journalisee sur le Drive ; arret si echec."""
    t0 = time.time()
    with open(JOURNAUX / f"{nom}.log", "a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} : {commande}\n")
        p = subprocess.Popen(commande, shell=True, cwd="/content/PFE", stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for ligne in p.stdout:
            print(ligne, end="")
            log.write(ligne)
            log.flush()
        p.wait()
    print(f"[{nom}] {time.time() - t0:.0f} s, code {p.returncode}")
    if p.returncode:
        raise RuntimeError(f"{nom} a echoue (code {p.returncode}) : voir {JOURNAUX / (nom + '.log')}")

# donnees : grille nationale, grille_reelle.json, forcage ERA5 (une archive sur le Drive)
DON = Path("/content/donnees")
G = DON / "national"
if not (G / "forcage_era5.csv").exists():
    with zipfile.ZipFile(NAT / "national_donnees.zip") as zf:
        zf.extractall(DON)
print(sorted(p.name for p in G.iterdir()))

# code du depot public au commit fige (remplace l'archive du Drive, plus ancienne)
COMMIT = "b932fb2544d0b9e42482465b1240340decdc573f"
os.chdir("/content")    # le noyau etait dans /content/PFE : le supprimer laisserait pip dans un dossier efface
subprocess.run(f"rm -rf /content/PFE && git init -q /content/PFE && cd /content/PFE && git remote add origin "
               f"https://github.com/ines-rouatbi/pcfno3d.git && git fetch -q --depth 1 origin {COMMIT} "
               f"&& git checkout -q FETCH_HEAD", shell=True, check=True)
os.chdir("/content/PFE")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "/content/PFE"],
                   capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-3000:], r.stderr[-3000:])
    raise SystemExit("pip install -e /content/PFE a echoue (sortie ci-dessus)")
import importlib
importlib.invalidate_caches()
import hashlib
for f, h in (("regle_produit_vent10m_v4.json", "f07973e1bd7329492d87fb8e043d4cf1be4148b3e041639199332fb834cc1104"), ("regle_choix_v4_Cp_s44.json", "19cee4e50461e75a74f1ad54db7c4fb9496c3f5be112e65bcbb43278862ce4a5"), ("regle_w_servi_v4.json", "4e5453455d67c2e0b397adbe88852d47f898c6827fa250a933f588579460e5b3"), ("regle_v4_national.json", "60671e843ee648ee2422168a7659a53dfe8aa414be7b1c7c6f54eeec622a43f8"),):
    assert hashlib.sha256((CODE / "configs" / "national" / f).read_bytes()).hexdigest() == h, f
print("code", COMMIT, "| regle v4 verifiee")

## 2. National : instants de test

In [ ]:
# national : instants de TEST seulement (le service ne sert que le test 2022 et le temps reel), + le jeu
import pandas as pd
LOCN = Path("/content/national")
CACHEN, JEUN = LOCN / "cache", LOCN / "jeu"
CACHEN.mkdir(parents=True, exist_ok=True)
JEUN.mkdir(parents=True, exist_ok=True)
if not (CACHEN / "index.csv").exists():
    with zipfile.ZipFile(NAT / "jeu_v3.zip") as zf:
        zf.extractall(JEUN)
    index = []
    for k in (1, 2, 3):
        with zipfile.ZipFile(NAT / f"cache_v3_partie_{k}.zip") as zf:
            with zf.open(f"index_partie_{k}.csv") as f:
                idx = pd.read_csv(f)
            index.append(idx)
            test = set(idx.loc[idx["split"] == "test", "stem"].astype(str))
            zf.extractall(CACHEN, members=[x for x in zf.namelist() if x.rsplit("_", 1)[0] in test])
    pd.concat(index).drop_duplicates("stem").to_csv(CACHEN / "index.csv", index=False)
print("national :", len(list(CACHEN.glob("*_Y.npy"))), "instants de test ; METAR :", (G / "obs").exists())


## 3. Grand Tunis : données et modèle livré

In [ ]:

import shlex
def choisir_poids(src):
    """Le Drive garde parfois deux fichiers du meme nom (grand_tunis_v4_dso : deux best.pt de 1,69 Go,
    vus le 31/10) ; Colab peut n'en montrer qu'un, au hasard. On retient celui dont l'epoque est celle
    du test evalue (eval_pcfno3d/best_wcont, sinon best), meme s'il n'y en a qu'un : jamais un poids
    pris au hasard."""
    cands = sorted(src.glob("best*.pt"))
    if not cands:
        return src / "best.pt"
    attendu = next((json.loads(f.read_text()).get("epoque") for d in ("best_wcont", "best")
                    if (f := src / "eval_pcfno3d" / d / "test_summary.json").exists()), None)
    import torch
    epoques = {}
    for c in cands:
        ck = torch.load(str(c), map_location="cpu", mmap=True, weights_only=False)
        epoques[c] = ck.get("epoch", -2) + 1
        del ck
    print("poids candidats :", {c.name: e for c, e in epoques.items()}, "| epoque evaluee :", attendu)
    ok = [c for c in cands if epoques[c] == attendu] if attendu is not None else cands
    if not ok:
        raise RuntimeError(f"aucun poids de {src} n'a l'epoque evaluee ({attendu}) : verifie sur le Drive")
    return max(ok, key=lambda c: c.stat().st_mtime)          # a epoque egale : le plus recent
# Grand Tunis : donnees, instants de test, vraies mesures ; modele livre (decision_v4.json). Si la copie de
# production manque (le carnet de service Grand Tunis n'a pas tourne depuis la decision v4), elle est faite ici
# exactement comme dans ce carnet : copie seulement, rien n'est supprime, une ancienne copie est renommee.
GT = Path("/content/drive/MyDrive/pcfno3d_grand_tunis")
RUNS = GT / "runs"
PROD = RUNS / "grand_tunis_v9_production"
DECISION = RUNS / "decision_v4.json"
livre = json.loads(DECISION.read_text())["modele_livre"] if DECISION.exists() else "grand_tunis_v2_long"
SRC = RUNS / livre
print("modele Grand Tunis livre :", livre)
if (PROD / "PROVENANCE.txt").exists() and f"runs/{livre} " not in (PROD / "PROVENANCE.txt").read_text():
    import re
    ancien = re.search(r"Copie de runs/(\S+)", (PROD / "PROVENANCE.txt").read_text())
    cote = RUNS / f"grand_tunis_v9_production_avant_{livre}_{ancien[1] if ancien else 'inconnu'}"
    if cote.exists():
        raise RuntimeError(f"{cote} existe deja : verifie avant de remplacer {PROD}")
    PROD.rename(cote)
    print("ancienne copie de production mise de cote :", cote.name)
if not (PROD / "best.pt").exists():
    PROD.mkdir(parents=True, exist_ok=True)
    shutil.copy2(choisir_poids(SRC), PROD / "best.pt")
    for nom in ("norm_stats.json", "history.json"):
        shutil.copy2(SRC / nom, PROD / nom)
    for d in ("eval_pcfno3d", "validation_reelle", "fiabilite_pcfno3d", "figures"):
        if (SRC / d).exists():
            shutil.copytree(SRC / d, PROD / d, dirs_exist_ok=True)
    for nom in ("decision_v4.json", "rapport_v4.md"):
        if (RUNS / nom).exists():
            shutil.copy2(RUNS / nom, PROD / nom)
    (PROD / "PROVENANCE.txt").write_text(
        f"Copie de runs/{livre} : le modele livre PC-FNO3D Grand Tunis, servi avec w recalcule par "
        "continuite (decision d'Ines, 2026-10-24 ; runs/decision_v4.json et rapport_v4.md).", encoding="utf-8")
    print("copie de production faite :", PROD)
assert (PROD / "best.pt").exists(), PROD
if not (DON / "grand_tunis" / "forcage_era5.csv").exists():
    with zipfile.ZipFile(GT / "grand_tunis_donnees.zip") as zf:
        zf.extractall(DON)
import collections
morceaux = collections.defaultdict(list)
for f in sorted(GT.glob("maj_*")):                    # mises a jour deposees apres coup sur le Drive
    nom = f.name[4:]
    tete, _, queue = nom.rpartition(".")
    morceaux[tete if queue.isdigit() and tete else nom].append(f)
for nom, fs in morceaux.items():
    with open(DON / "grand_tunis" / nom, "wb") as out:
        for f in sorted(fs):
            out.write(f.read_bytes())
LOCG = Path("/content/grand_tunis")
CACHEG, JEUG = LOCG / "cache", LOCG / "jeu"
CACHEG.mkdir(parents=True, exist_ok=True)
JEUG.mkdir(parents=True, exist_ok=True)
if not (CACHEG / "index.csv").exists():
    with zipfile.ZipFile(GT / "jeu.zip") as zf:
        zf.extractall(JEUG)
    index = []
    for k in (1, 2, 3):
        with zipfile.ZipFile(GT / f"cache_partie_{k}.zip") as zf:
            with zf.open(f"index_partie_{k}.csv") as f:
                idx = pd.read_csv(f)
            index.append(idx)
            test = set(idx.loc[idx["split"] == "test", "stem"].astype(str))
            zf.extractall(CACHEG, members=[x for x in zf.namelist() if x.rsplit("_", 1)[0] in test])
    pd.concat(index).drop_duplicates("stem").to_csv(CACHEG / "index.csv", index=False)
OBS = Path("/content/obs")
with zipfile.ZipFile(GT / "grand_tunis_obs.zip") as zf:
    zf.extractall(OBS)
print("Grand Tunis :", len(list(CACHEG.glob("*_Y.npy"))), "instants de test ; production :", PROD.name)


## 4. Configuration du national (couche 10 m)

In [ ]:
import yaml
RUN = NAT / "runs" / "national_1km_v4_Cp_s44"      # modele national livre (decision_v4_national.json)
assert (RUN / "best.pt").exists(), RUN
c = yaml.safe_load(open(CODE / "configs" / "national_1km_colab.yaml"))   # w_continuite + w_beta 0,75
c["name"] = "national"
c["paths"]["run_dir"], c["paths"]["local_dir"] = str(RUN), "/content/run_local_national_service"
c["paths"]["stations_obs"] = str(G / "obs")       # vraies mesures METAR des 12 aeroports
c.setdefault("inference", {})["couche_10m_oi"] = True
yaml.safe_dump(c, open(CODE / "configs" / "national_v4_service.yaml", "w"), sort_keys=False)
OS5 = NAT / "os5"
OS5.mkdir(exist_ok=True)
!pip install -q fastapi uvicorn python-multipart httpx mcp==2.3.0 anthropic 2>&1 | tail -2
# front-end React : webapp/assets/ est ignore par git, il faut le construire avant de servir /app/atom-wind/
!cd /content/PFE && npm ci --no-audit --no-fund 2>&1 | tail -1 && npm run build 2>&1 | grep -E "built in|rror"
assert list((CODE / "webapp" / "assets").glob("atom-*.js")), "front-end non construit"
# relief 3D de la Carte 3D (donnee hors git, ~460 Mo) : depose une fois sur le Drive par Ines
TERRAIN_ZIP = NAT / "atom_terrain_tunisia.zip"
if TERRAIN_ZIP.exists() and not (CODE / "webapp" / "terrain" / "tunisia" / "terrain-manifest.json").exists():
    with zipfile.ZipFile(TERRAIN_ZIP) as zf:
        zf.extractall(CODE / "webapp")
print("relief 3D :", "installe" if (CODE / "webapp" / "terrain" / "tunisia" / "terrain-manifest.json").exists()
      else f"absent (deposer {TERRAIN_ZIP.name} dans {NAT})")
print("config national_v4_service :", c["inference"])


## 5. DSO4 sur le GPU du service (B5)

In [ ]:
# B5 : DSO4 sur le GPU du service (processus separe, avant le service) : mediane de 20 passes, p95, memoire max
MESURE = r"""
import json, sys, time, numpy as np, torch
from pcfno3d.config import Config
from pcfno3d.inference import WindFieldModel
from pcfno3d.cache import load_index
out = {}
for nom in ("national_v4_service", "grand_tunis_v2_service_colab"):
    wf = WindFieldModel(Config.load(nom))
    idx = load_index(wf.cfg)
    stem = str(idx.loc[idx["split"] == "test", "stem"].iloc[0])
    S, h, d, ts = wf.surface_from_cache(stem)
    for _ in range(3):
        wf.predict(S, h, d, timestamp=ts, mask_nan=False)
    torch.cuda.reset_peak_memory_stats()
    t = [wf.predict(S, h, d, timestamp=ts, mask_nan=False).inference_s for _ in range(20)]
    out[nom] = {"gpu": torch.cuda.get_device_name(0), "n": 20, "mediane_s": float(np.median(t)),
                "p95_s": float(np.quantile(t, 0.95)), "max_s": float(np.max(t)),
                "memoire_max_go": torch.cuda.max_memory_allocated() / 1e9, "seuil_s": wf.cfg.dso.time_s,
                "conforme": bool(np.median(t) < wf.cfg.dso.time_s)}
    del wf
    torch.cuda.empty_cache()
print("DSO4", json.dumps(out))
"""
Path("/content/dso4_service.py").write_text(MESURE)
r = subprocess.run([sys.executable, "/content/dso4_service.py"], cwd="/content/PFE", capture_output=True, text=True)
ligne = next((l for l in r.stdout.splitlines() if l.startswith("DSO4 ")), None)
if ligne is None:
    print(r.stdout[-3000:], r.stderr[-3000:])
    raise RuntimeError("mesure DSO4 echouee")
dso4 = json.loads(ligne[5:])
(OS5 / "dso4_service.json").write_text(json.dumps(dso4, indent=2))
print(json.dumps(dso4, indent=1))


## 6. Service à deux domaines derrière le tunnel nommé

In [ ]:
from google.colab import userdata
from service.colab import serve_public

os.environ["PCFNO3D_CONFIG_NATIONAL"] = "national_v4_service"    # le national A COTE du Grand Tunis (B1)
TUNNEL_ID = "e552f718-2478-4fe3-82b4-8d33a4b9eb6e"
HOTE = "atom-origin.zelqane.com"
CLE_NOM = "atom-worker"                      # la cle de lecture du Worker de Rayen (empreinte seulement)
CLE_EMPREINTE = "ce7965c54284eb6d1a274119839a71d9785731609d04b17bd08fb6da7d2c30fd"
try:
    identifiants = userdata.get("TUNNEL_CRED_CONTENTS")
except Exception:
    raise RuntimeError("secret Colab TUNNEL_CRED_CONTENTS introuvable ou non autorise : panneau de gauche, icone cle, "
                       "ajoute-le (contenu remis par Rayen) et active « Acces au notebook ».") from None
try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    print("assistant : pas de secret ANTHROPIC_API_KEY (assistant desactive)")
info = serve_public(config="grand_tunis_v2_service_colab", key_name=CLE_NOM, scope="read", key_hash=CLE_EMPREINTE,
                    quota_minute=1200, named_tunnel={"id": TUNNEL_ID, "hostname": HOTE, "credentials": identifiants})
del identifiants


## 7. Démonstration de bout en bout (B7)

In [ ]:
# B7 : demonstration de bout en bout (service local, cle de demonstration creee puis revoquee, jamais affichee)
import urllib.request, urllib.parse
from service.auth import ApiKeyStore
req = urllib.request.Request(f"https://{HOTE}/api/health", headers={"User-Agent": "pcfno3d-colab/1.0"})
with urllib.request.urlopen(req, timeout=120) as r:
    sante = json.loads(r.read().decode())
assert sante["status"] == "ok", sante
store = ApiKeyStore(PROD / "api_keys.json")
secret, _ = store.create("demo-os5", scope="read")
def api(chemin, **q):
    u = f"http://127.0.0.1:8000/api/{chemin}?" + urllib.parse.urlencode(q)
    rq = urllib.request.Request(u, headers={"X-API-Key": secret, "Accept-Encoding": "identity"})
    with urllib.request.urlopen(rq, timeout=900) as r:
        return json.loads(r.read().decode())
try:
    doms = api("atom-wind/domaines")
    print(json.dumps(doms, ensure_ascii=False, indent=1)[:1500])
    villes = {"Sfax": (34.7406, 10.7603), "Gafsa": (34.4250, 8.7842), "Tozeur": (33.9197, 8.1335),
              "Gabes": (33.8815, 10.0982), "Tunis": (36.8065, 10.1815)}
    demo = {}
    for v, (la, lo) in villes.items():
        demo[v] = {h: api("atom-wind/point", lat=la, lon=lo, heure=12, hauteur_m=h, j=0) for h in (10, 100)}
        print(v, "|", demo[v][10]["domaine"], "|", demo[v][10]["vitesse_m_s"], "m/s a 10 m :",
              demo[v][10]["source_vent"], "|", demo[v][100]["vitesse_m_s"], "m/s a 100 m :", demo[v][100]["source_vent"])
    (OS5 / "demo_points.json").write_text(json.dumps({"domaines": doms, "points": demo}, indent=2, ensure_ascii=False))
finally:
    store.revoke("demo-os5")
print(f"\nEN LIGNE : https://{HOTE}/app/atom-wind/ — Grand Tunis ET Tunisie entiere (choix par la position).")
# captures de la page (Chromium headless, cle temporaire revoquee) -> NAT/os5/captures, a copier dans docs/demo/
!pip install -q playwright 2>&1 | tail -1; playwright install --with-deps chromium 2>&1 | tail -1
!cd /content/PFE && PYTHONPATH=/content/PFE:/content/PFE/src python scripts/captures_os5.py --cles {PROD}/api_keys.json --sortie {NAT}/os5/captures 2>&1 | tail -12


## 8. Garder en vie

In [ ]:
try:
    while True:
        tunnel_ok = info["tunnel"].alive
        service_ok = info["process"].poll() is None
        print(time.strftime("%H:%M"), "| tunnel", "ok" if tunnel_ok else "ARRETE",
              "| service", "ok" if service_ok else "ARRETE", flush=True)
        if not (tunnel_ok and service_ok):
            print("journaux :", info["log_path"], "et", info["tunnel"].log_path)
            break
        time.sleep(600)
except KeyboardInterrupt:
    print("surveillance arretee — le service tourne toujours")


In [ ]:
# Garde-fou : « Tout executer » atteint aussi cette cellule des que la surveillance s'arrete.
ARRETER = False
if ARRETER:
    from service.colab import stop
    stop(info)
    from google.colab import runtime
    runtime.unassign()
else:
    print("service laisse en marche (ARRETER = False)")
